# 1. From deformation to energy and stress

**Goal:** write a material density in ordinary Torch and differentiate it. No FEM objects yet.

A body has many material points. At each point, the $3\times3$ deformation gradient
$F$ describes how small vectors change. $F=I$ is rest; $F=\operatorname{diag}(s,1,1)$
stretches the first direction by $s$. We use dimensionless geometry and consistent
arbitrary units throughout this series. Plasticity is fixed to $F_p=I$.

We will distinguish **density** $W$ (energy per reference volume) from **total** energy $E$.
Run every cell in order using the project's Python environment. You can restart the kernel
and run this notebook independently of all other lessons.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

torch.set_num_threads(1)
dtype = torch.float64

## Write the constitutive law

Use a compressible Neo-Hookean density with parameters $\theta=(\lambda,\mu)$:

$$W(F,\theta)=\frac{\mu}{2}(\|F\|^2-3)-\mu\log J+\frac{\lambda}{2}(\log J)^2,
\qquad J=\det F>0.$$

The inputs are `F: (N,3,3)` and `theta: (N,2)`. Output `W: (N,)` contains one
independent value per row. Do not average over the material-point axis inside the law.
Here we evaluate only positive-determinant states and positive material parameters.

In [ ]:
def density(F, theta):
    lam, mu = theta.unbind(-1)
    log_J = torch.linalg.det(F).log()
    return 0.5 * mu * (F.square().sum((-2, -1)) - 3) - mu * log_J + 0.5 * lam * log_J.square()

F = torch.diag(torch.tensor([1.1, 1., 1.], dtype=dtype))[None].requires_grad_()
theta = torch.tensor([[2., 3.]], dtype=dtype)
W = density(F, theta)
P = torch.autograd.grad(W.sum(), F)[0]
print('density:', W.item())
print('first Piola stress:\n', P.detach().numpy()[0])

## Why differentiate a sum?

$P_{ij}=\partial W/\partial F_{ij}$ is first Piola stress. Since rows do not depend
on each other, differentiating `W.sum()` with respect to `F[n]` returns exactly the
derivative of `W[n]`. This is a batching convenience, not integration over a mesh.

Check $P_{11}$ by perturbing only $F_{11}$. The central difference should agree
with autograd; reducing the step indefinitely eventually amplifies roundoff.

In [ ]:
h = 1e-6
step = torch.zeros_like(F)
step[0, 0, 0] = h
finite_difference = (density(F.detach() + step, theta) - density(F.detach() - step, theta)) / (2 * h)
torch.testing.assert_close(P[0, 0, 0], finite_difference[0], rtol=1e-6, atol=1e-9)
rest = torch.eye(3, dtype=dtype)[None].requires_grad_()
rest_W = density(rest, theta)
rest_P = torch.autograd.grad(rest_W.sum(), rest)[0]
assert rest_W.abs().max() < 1e-12 and rest_P.abs().max() < 1e-12
print('stress finite-difference error:', abs(P[0, 0, 0] - finite_difference[0]).item())

## Observe a whole loading path

Now evaluate many independent stretches together. The identity state has zero
energy and stress; compression and extension both cost energy.

In [ ]:
stretch = torch.linspace(0.8, 1.2, 81, dtype=dtype)
path = torch.eye(3, dtype=dtype).repeat(len(stretch), 1, 1)
path[:, 0, 0] = stretch
path.requires_grad_()
values = density(path, theta.expand(len(stretch), -1))
stress = torch.autograd.grad(values.sum(), path)[0][:, 0, 0]
fig, axes = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)
axes[0].plot(stretch.numpy(), values.detach().numpy())
axes[1].plot(stretch.numpy(), stress.detach().numpy())
for ax, label in zip(axes, ['Energy density W', 'First Piola stress P11']):
    ax.set(xlabel='Stretch s', ylabel=label)
    ax.axvline(1, color='gray', linestyle=':')
    ax.grid(alpha=0.25)
plt.show()

## Try it

Double both material parameters and rerun the loading path. Do energy and stress
double? Then change only one parameter and compare compression with extension.

**Checkpoint:** you can now write a row-independent density and check its first derivative.
Next: put the same law into native FEM in lesson 2.